# Chapter 4 (part 3) — Interpreting the Regression Equation

Covers: Correlated Predictors → Multicollinearity → Confounding Variables → Interactions and Main Effects, on Ames Housing, plus a short supplementary section on mediators.

The thread: **a regression coefficient is a conditional statement, "the difference in y per unit of x among houses that look the same on everything else in the model", so what's *in* the model changes what every coefficient means.** Correlated predictors leave few houses that differ on one but not the other. Confounding means a missing variable is silently doing the work. Interactions mean "what's a square foot worth?" depends on who you ask. This notebook leans toward causal-inference questions.

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

pd.set_option("display.max_columns", None)

from mlstats.data import load_ames
ames = load_ames()
ames.shape

(2930, 81)

## 1. Correlated Predictors and Multicollinearity

Two predictors that move together both "explain" the same variation in y; once one is in the model, the other only gets credit for what's *left over*. Numeric picture: bigger houses have more bedrooms and cost more. Alone, a bedroom appears to add a lot of value, because it's standing in for square footage. Hold size fixed and a 4-bedroom house has smaller rooms than a 3-bedroom one.

The **variance inflation factor (VIF)** measures how much of predictor j the other predictors already explain: regress j on the others, take that R², and compute `VIF = 1 / (1 - R²)`. If the others explain 90% of j, VIF = 1/(1-0.9) = 10: its coefficient's variance is ten times what it would be with uncorrelated predictors (standard error ~3.2× bigger).

**Predict first:** when you add `Gr_Liv_Area`, will the bedroom coefficient shrink, vanish, or flip sign? Of `Gr_Liv_Area, First_Flr_SF, Total_Bsmt_SF, Year_Built, Garage_Area`, which two will have the highest VIF?

**Question 1.** (a) Fit `Sale_Price ~ Bedroom_AbvGr`, then `Sale_Price ~ Bedroom_AbvGr + Gr_Liv_Area`. Explain the change using the mean `Gr_Liv_Area` by bedroom count. What comparison does the second coefficient describe? (b) For the five predictors above, compute the VIF of `Gr_Liv_Area` by hand (auxiliary regression), then all five with `variance_inflation_factor`. Which are inflated, and why those?

*Tools for this question:*
- `df.groupby("Bedroom_AbvGr")["Gr_Liv_Area"].mean()`; read one coefficient with `res.params["Bedroom_AbvGr"]` and `res.bse["Bedroom_AbvGr"]`.
- `variance_inflation_factor(X.values, i)` (from `statsmodels.stats.outliers_influence`): `X` should be the predictor DataFrame **with** `sm.add_constant(...)`, and `i` is the column position to score.
- By hand: regress predictor j on the others, `1 / (1 - r2)`.

_Prediction (one line, before running anything):_

* I believe that controlling for living area will most likely flip the sign for the bedroom coefficient. I believe that its because homes of the same size but one that has more bedrooms implies that those bedrooms may be smaller. 
* I believe that a model that only contains bedroom may be using that variable as a potential stand-in for home sq footage. So living area and first floor square feet may already capture/explain the majority of the variance of what the bedroom predictor was trying to capture before.

In [8]:
bedroom_model = smf.ols("Sale_Price ~ Bedroom_AbvGr", data = ames).fit()
print(f"Coefficient: {bedroom_model.params["Bedroom_AbvGr"]}")
print(f"Standard error: {bedroom_model.bse["Bedroom_AbvGr"]}")

Coefficient: 13889.495269004663
Standard error: 1765.0419433748586


In [9]:
bedroom_liv_area_model = smf.ols("Sale_Price ~ Bedroom_AbvGr + Gr_Liv_Area", data = ames).fit()
print(f"Gr_Liv_Area Coefficient: {bedroom_liv_area_model.params["Gr_Liv_Area"]}")
print(f"Bedroom Count Coefficient: {bedroom_liv_area_model.params["Bedroom_AbvGr"]}")
print(f"Standard error: {bedroom_liv_area_model.bse["Bedroom_AbvGr"]}")

Gr_Liv_Area Coefficient: 136.36090297508093
Bedroom Count Coefficient: -29149.109660217087
Standard error: 1372.1351897077122


In [10]:
ames.groupby("Bedroom_AbvGr")["Gr_Liv_Area"].mean()

Bedroom_AbvGr
0    1364.000000
1    1155.080357
2    1224.430686
3    1481.261115
4    2043.947500
5    2351.020833
6    2126.809524
8    3395.000000
Name: Gr_Liv_Area, dtype: float64

* For the most part, as bedroom count increases, so does the size of the living area. The first model was trying to fit this information into one coefficient. But the second model controls for each variable separately, so we can now see the effect of changes in living area and bed room count individually.
* The second coefficient descibes the change in average sales price per sq foot of gr_liv_area on homes with the same number of rooms.
* The bedroom coefficient in the second model answers a different comparison: among homes with the **same living area**, one extra bedroom goes with about **$29k lower** price (-29,149). That is not a causal penalty on bedrooms. Same total space split into more rooms means smaller rooms, which buyers value less.
* Rough numbers behind the flip: going from 2 to 3 bedrooms adds ~257 sq ft on average (1,224 to 1,481), and 3 to 4 adds ~563 (to 2,044). At $136/sq ft that is ~$35k to ~$77k of price that the first model credited to the extra bedroom. Once area is held fixed, the bedroom coefficient only gets what is left over, and that is negative.

In [11]:
formula = "Gr_Liv_Area ~ First_Flr_SF + Total_Bsmt_SF + Year_Built + Garage_Area"

aux_model = smf.ols(formula, data=ames).fit()

r_squared_aux = aux_model.rsquared

vif_gr_liv_area = 1 / (1 - r_squared_aux)

print(f"Auxiliary R² for Gr_Liv_Area: {r_squared_aux:.4f}")
print(f"Hand-computed VIF for Gr_Liv_Area: {vif_gr_liv_area:.4f}")

Auxiliary R² for Gr_Liv_Area: 0.3758
Hand-computed VIF for Gr_Liv_Area: 1.6020


In [12]:
data = ames[["Gr_Liv_Area","First_Flr_SF","Total_Bsmt_SF","Year_Built", "Garage_Area"]]

X = sm.add_constant(data)
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# 4. Filter out the constant row to view your features
print(vif_data[vif_data["Feature"] != "const"])


         Feature       VIF
1    Gr_Liv_Area  1.602025
2   First_Flr_SF  3.310867
3  Total_Bsmt_SF  3.046530
4     Year_Built  1.387921
5    Garage_Area  1.708464


* The 2 variables that had a more inflated VIF was the first floor square foot and total basement square foot. This indicate that these 2 predictors have mild correlation with another independent variable. To me, this might make sense for first floor SF, since the larger the Gr_liv_area (living area which is usually on the first floor of a home) the larger the first floor is in general. 
* Now as for basement SF, the only way I can think about it is that generally a basement sitting below first floor has to be around the same size as the floor above it? So as the first floor size increases, so does the basement size?

_What matched, what surprised you, and why (2-3 bullets):_
* For the most part, my inital predictions regarding the addition of the predictor matched up.
* Seems like bedroom was trying to capture some sort of information the gr_liv_area was capturing becuase as soon as we added the new variable it flipped signs.
* The standard error on the bedroom coefficient also dropped (1,765 to 1,372) while the sign flipped, so the flip is not noise from collinearity. Bedroom and living area are correlated, but the VIFs are mild, so this is a change in what the coefficient *means*, not an instability problem.
* On VIF: the top two were `First_Flr_SF` (3.31) and `Total_Bsmt_SF` (3.05), which matched what I expected. Even those are mild (well under the usual 5-10 worry line). `Gr_Liv_Area` is only 1.60 here because `Second_Flr_SF` is not among the predictors, even though `Gr_Liv_Area = First_Flr_SF + Second_Flr_SF + Low_Qual_Fin_SF` exactly. VIF only sees the predictors in the model.

## 2. Confounding and the Omitted-Variable-Bias Identity

A **confounder** drives both the variable you care about and the outcome. Numeric picture: houses with central air sell for $60k more on average. But AC is much more common in newer houses, and old houses are cheap for many reasons unrelated to AC. Compare only houses of similar age and the AC premium shrinks.

*Supplementary (not in the book):* there's an exact accounting rule behind this. Long model `y ~ b*x + c*z`, short model `y ~ b'*x`. Then **b' = b + c × δ**, where δ is the slope from regressing z on x. Numeric picture: AC is worth $13k in the long model, each year newer is worth $1,000, and AC houses are 25 years newer on average (δ = 25). The short model's AC coefficient is 13,000 + 1,000 × 25 = $38,000. This is an identity of least squares, not an approximation.

**Predict first:** as you add `Year_Built` and then `Gr_Liv_Area`, will the `Central_Air` coefficient shrink or grow?

**Question 2.** (a) Track the `Central_Air` coefficient and standard error across (i) `Central_Air` only, (ii) `+ Year_Built`, (iii) `+ Gr_Liv_Area`. (b) Models (i) and (ii) are exactly the short and long models with z = `Year_Built`. Fit the one auxiliary regression `Year_Built ~ Central_Air` and verify the identity to the dollar. (c) Is the model-(iii) coefficient "the value of installing AC"? Name one confounder you can't observe in this dataset and the direction it would bias the estimate.

*Tools for this question:*
- 0/1 treatment column: `(df["Central_Air"] == "Y").astype(int)`. With `C(Central_Air)` instead, the coefficient is named `C(Central_Air)[T.Y]`.
- Collect results with `res.params[name]`, `res.bse[name]` in a list, then `pd.DataFrame`.

_Prediction (one line, before running anything):_
* I believe that adding more variables after the initial central_ac will shrink its coefficient. Having an equation with only AC and sale price as the variables will try to fit all the information it can about a house into one variable. However, as we add more variables and can begin identifying patterns/distinctions, the affect of AC on a home will likely decrease.

In [16]:
# central air model
model = smf.ols("Sale_Price ~ Central_Air", data = ames).fit()
print("Sale_Price ~ Central_Air")
print(f"Coefficients:")
print(f"{model.params}\n")
print("Standard Errors")
print(f"{model.bse}\n")

# AC with year built
model = smf.ols("Sale_Price ~ Central_Air + Year_Built", data = ames).fit()
print("Sale_Price ~ Central_Air + Year_Built")
print(f"Coefficients:")
print(f"{model.params}\n")
print("Standard Errors")
print(f"{model.bse}\n")

# AC with year built and gr_liv_area
model = smf.ols("Sale_Price ~ Central_Air + Year_Built + Gr_Liv_Area", data = ames).fit()
print("Sale_Price ~ Central_Air + Year_Built + Gr_Liv_Area")
print(f"Coefficients:")
print(f"{model.params}\n")
print("Standard Errors")
print(f"{model.bse}\n")

Sale_Price ~ Central_Air
Coefficients:
Intercept           101890.479592
Central_Air[T.Y]     84562.308265
dtype: float64

Standard Errors
Intercept           5503.899800
Central_Air[T.Y]    5697.772095
dtype: float64

Sale_Price ~ Central_Air + Year_Built
Coefficients:
Intercept          -2.618714e+06
Central_Air[T.Y]    2.112467e+04
Year_Built          1.410095e+03
dtype: float64

Standard Errors
Intercept           84070.912971
Central_Air[T.Y]     5265.835907
Year_Built             43.505318
dtype: float64

Sale_Price ~ Central_Air + Year_Built + Gr_Liv_Area
Coefficients:
Intercept          -2.038232e+06
Central_Air[T.Y]    1.362585e+04
Year_Built          1.046359e+03
Gr_Liv_Area         9.573077e+01
dtype: float64

Standard Errors
Intercept           60158.922914
Central_Air[T.Y]     3711.207589
Year_Built             31.357553
Gr_Liv_Area             1.755134
dtype: float64



* **How I interpret what model 3 represents**: it seems that homes with central AC on average have a sales price of 13625.85 but it doesn't necessarily say that impact of having it installed on a home that doesnt already have it.

* **Possible missing confounder** : I recognize that confounders are variables that affect both dependent and independent variable. So a variable that we did not consider that could effect this could be outdoor physical space. Some homes with less physical space may simply not be able to install central AC. On the other side, more outdoor space can directly imply a increase in sales price.
* Correction to the model 3 reading: $13,626 is not the average price of AC homes. It is the price gap between AC and no-AC homes **with the same `Year_Built` and `Gr_Liv_Area`**. It still is not the value of installing AC, because the two groups can differ in ways the model doesn't see.
* Correction on the confounder: outdoor space is observed here (`Lot_Area`, `Wood_Deck_SF`, `Pool_Area`), so it isn't a variable I can't observe. It would just be a control to add. An unobservable one is **owner upkeep / interior condition**. Owners who maintain their homes are more likely to have working central air, and their homes sell for more, so the AC coefficient is biased **upward**: part of the $13.6k is really 'well-kept house', not AC.

In [18]:
# very identity to dollar with Year_Built ~ Central_Air
model = smf.ols("Year_Built ~ C(Central_Air)", data = ames).fit()
print(model.params)
print(model.bse)


Intercept              1929.377551
C(Central_Air)[T.Y]      44.988213
dtype: float64
Intercept              2.005954
C(Central_Air)[T.Y]    2.076613
dtype: float64


* In terms of the regression with Year_Built ~ Central_Air, seems to potentially show that homes with AC are on average 44 years newer with the intercept being 1929?
* Identity check (do this in code too): short coef = long coef + (Year_Built coef in long) x (AC coef in the auxiliary). 21,124.67 + 1,410.09 x 44.99 = **84,562.31**, which matches the short model's 84,562.31 to the cent. About **$63k of the original $84.6k** is just house age, leaving $21k, and then $13.6k once size is held fixed.
* The intercept 1929 is the average build year of the no-AC homes, and the 45 is the gap, so AC homes average about 1974. The no-AC group is small (196 homes vs 2,734), so the 1929 average is itself noisy (standard error ~2 years).

_What matched, what surprised you, and why (2-3 bullets):_
* My initial predictions were correct for the most part regarding how adding variables on top of AC decreases its coefficient.
* Adding the additional variables helped improve the standard errors and indicating that we were trying to fit too much information into one variable.
* Correction on the standard errors: the AC standard error fell (5,698 to 5,266 to 3,711) because `Year_Built` and `Gr_Liv_Area` explain a lot of price, which shrinks the residual noise and so the standard error. That is separate from the bias: the *coefficient* shrank because AC was standing in for house age, and the standard error shrank because the model fits better. One is about bias, the other about precision.

## 3. Interactions and Main Effects

An **interaction** lets one variable's effect depend on another's. Numeric picture: if price rises $50 per sqft for homes without central air and $110 per sqft with it, "the effect of a square foot" isn't one number. The model `y ~ x * g` includes `x`, `g`, and the product `x:g`. The coefficient on `x` is now the slope **for the reference group of g only**, the coefficient on `g` is the gap **at x = 0**, and the product term is the *difference in slope* between groups.

**Predict first:** will the price-per-sqft slope be steeper for AC or no-AC homes? Will the `Central_Air[T.Y]` coefficient itself be positive or negative?

**Question 3.** (a) Fit `Sale_Price ~ Gr_Liv_Area * C(Central_Air)` and write the fitted line for each group as an equation with an explicit slope and intercept. (b) Explain the sign of the `Central_Air[T.Y]` coefficient in terms of what `Gr_Liv_Area = 0` means. What would centering `Gr_Liv_Area` do to it? (No need to refit.) (c) How many no-AC homes are there, and how precisely is their slope estimated?

*Tools for this question:*
- `"y ~ x * C(g)"` expands to `x + C(g) + x:C(g)`. The product coefficient is named like `Gr_Liv_Area:C(Central_Air)[T.Y]`.

_Prediction (one line, before running anything):_
* My prediction is that the slope for homes with AC will be steeper since I believe the sq ft of homes with AC will be more valuable than the ones without. I believe the coefficient of Central AC itself will be positive.

In [21]:
# fit the model
model = smf.ols("Sale_Price ~ Gr_Liv_Area * C(Central_Air)", data = ames).fit()
print(model.params)
print(model.bse)

Intercept                          39114.845676
C(Central_Air)[T.Y]               -23011.590472
Gr_Liv_Area                           49.542882
Gr_Liv_Area:C(Central_Air)[T.Y]       62.797834
dtype: float64
Intercept                          10558.015831
C(Central_Air)[T.Y]                11060.595113
Gr_Liv_Area                            7.753301
Gr_Liv_Area:C(Central_Air)[T.Y]        8.023253
dtype: float64


In [22]:
ames["Central_Air"].value_counts()

Central_Air
Y    2734
N     196
Name: count, dtype: int64

**Fitted Line Per Group Breakdown**

***No AC***
* Intercept remains at 39114.85 while the Gr_Liv_Area coefficient is 49.54.

***With AC***
* Intercept remains 39114.85 and Gr_Liv_Area remains at 49.54.
* Central AC as a binary variable on its own seems to reduce the average sale price by 23k but that is most likely due to the fact that the interaction variable captures the difference/increase in average sales price with the coefficient being 62.80. My interpretation of this is almost like a premium/added value onto each square foot of the home with central AC.
* Correction to the AC line: the AC line is not the base intercept and slope. Add the AC terms. Intercept = 39,115 + (-23,012) = **16,103**. Slope = 49.54 + 62.80 = **112.34**. So no-AC: `price = 39,115 + 49.5 x area`, AC: `price = 16,103 + 112.3 x area`.

**Coefficient of Central AC Explanation**
* Central AC's coefficient seemed to be negative with the interaction variable which may seem strange since when you put it in terms of a home with 0 sq ft of living area, the presence of central AC decreases the average sales price of that home by 23k.
* If Gr_Liv_Area was centered and having x=0, I would assume that the coefficient of Central AC would decrease since it would be relative to an average home and not a home with 0 sq ft which is what the current model uses as a reference.

**No-AC Homes**
* The balance between homes with/without central air is heavily inbalancd with majority having central AC. We can see its impact on the model as its the variable with the highest standard error.
* Count and precision: there are **196** no-AC homes vs 2,734 with AC (7%). The no-AC slope is 49.5 with standard error 7.8, so a 95% interval of roughly 34 to 65. That is wide next to the gap between the two slopes (62.8).
* Correction on centering: the `Central_Air[T.Y]` coefficient is the AC-vs-no-AC gap *at x = 0*. The AC line is steeper, so the gap grows with size. Centering at the mean (~1,500 sq ft) moves where the gap is measured, so the dummy coefficient would **rise** from -23,012 to about -23,012 + 62.8 x 1,500 = **+$71k** (not fall). The slopes and the fitted lines don't change, only what the dummy coefficient reports.

_What matched, what surprised you, and why (2-3 bullets):_
* While homes with AC, based on the equation, could have a higher sale price on average, I was wrong regarding the sign of the coefficient for central AC. I did not account that if the central AC term was positive the interaction variable would either be more value added on top of that or there would be nothing representing the relationship/effect of homes with no AC.
* The negative sign isn't because nothing else represents no-AC homes (the base `Gr_Liv_Area` slope and intercept already are the no-AC line). It comes from the lines crossing: the AC line starts lower at area 0 (16,103 vs 39,115) but is steeper (112.3 vs 49.5 per sq ft), so they cross at 23,012 / 62.8 = **~366 sq ft**. Nearly every home is bigger than that, so AC is higher for almost all real homes.

## 4. Supplementary: Controls That Hurt (Mediators)

*(Beyond the book, but the natural follow-up to confounding.)* If a variable sits *between* the cause and the effect (a **mediator**), controlling for it blocks the pathway you're trying to measure. Numeric picture: a remodel works partly by raising overall quality, which buyers pay for. Control for quality and you're comparing remodeled and unremodeled homes *of the same quality*, which wipes out the main way remodeling helps.

Data quirk: `Year_Remod_Add` is floored at 1950, so for older homes `Year_Remod_Add > Year_Built` is true whether or not anything was remodeled. Restrict to `Year_Built >= 1950` and define `remod = (Year_Remod_Add > Year_Built)`.

**Predict first:** when you add overall quality to the model, will the `remod` coefficient go up, go down, or stay about the same?

**Question 4.** (a) Fit `Sale_Price ~ remod`, then `+ Year_Built + Gr_Liv_Area`, then `+ numeric overall quality` (reuse your 1-10 mapping from the factor-variables notebook). Report the `remod` coefficient each time. (b) Which one would you report as "the effect of remodeling on price", and why is the last one the wrong answer to that question?

*Tools for this question:*
- `(df["Year_Remod_Add"] > df["Year_Built"]).astype(int)` for the flag; `df.query("Year_Built >= 1950")` to restrict.

_Prediction (one line, before running anything):_
* I believe that when you add overall quality to the regression, the coefficient of the remod binary will decrease since its original value came from identfiying the quality of a home based on whether or not it was remodeled. After controlling for quality, the signal that remod was trying to capture will be diminished.

In [30]:
remod_df = ames.copy()
remod_df = remod_df[remod_df["Year_Built"] >= 1950]
remod_df["remod"] = np.where(remod_df["Year_Remod_Add"] > remod_df["Year_Built"], 1, 0)

In [31]:
# build quality map
qual_map = {
    "Very_Poor": 1,
    "Poor": 2,
    "Fair": 3,
    "Below_Average": 4,
    "Average": 5,
    "Above_Average": 6,
    "Good": 7,
    "Very_Good": 8,
    "Excellent": 9,
    "Very_Excellent": 10,
}

# map the qualities
remod_df["Quality_Tier"] = ames["Overall_Qual"].map(qual_map)

In [32]:
# fit the first model
model = smf.ols("Sale_Price ~ remod", data = remod_df).fit()
print(model.params)

Intercept    184324.679618
remod         34256.588239
dtype: float64


In [34]:
# fit model with year built and living area 
model = smf.ols("Sale_Price ~ remod + Year_Built + Gr_Liv_Area", data = remod_df).fit()
print(model.params)


Intercept     -2.763183e+06
remod          1.138694e+04
Year_Built     1.416125e+03
Gr_Liv_Area    9.516143e+01
dtype: float64


In [35]:
# fit model with numeric overall quality
model = smf.ols("Sale_Price ~ remod + Year_Built + Gr_Liv_Area + Quality_Tier", data = remod_df).fit()
print(model.params)

Intercept      -377437.277892
remod             9790.914633
Year_Built         136.966482
Gr_Liv_Area         62.798935
Quality_Tier     31843.119715
dtype: float64


* I believe the second model would be better to answer the question of how remodeling affects sales price. It controls for confounders that might make the estimate unstable and is better than the first model that tries to cram in all the information into one coefficient. 
* If we went with the second model, the coefficient would be saying that, on average, homes that were remodeled saw 11386.94 higher sales price.
* Quality tier model may be wrong for 2 main reasons in my mind. For one, quality tier implies that the change in sales price is linear as tiers move up and change and this is not the case which means that the model's estimates might not be the most stable. On the otherhand, the whole point of the remodel variable was to try to capture the idea that remodeled homes imply a higher quality right? So by controlling for quality, the remodeling term no longer captures that implication of higher quality and therefor higher price. Thats why I believe the coefficient for remod went down further after controlling for that.
* Why model 2: `Year_Built` and `Gr_Liv_Area` are plausible confounders (they affect whether and how a house gets remodeled, and they affect price), so we compare remodeled and unremodeled homes of the same age and size. Corrected reading of the coefficient: *among homes with the same build year and size, remodeled ones sell for about **$11k more*** (an association, not what a given house 'saw' change).
* The mediator point is the real reason model 3 answers the wrong question: it compares remodeled and unremodeled homes of the *same quality*, so it only keeps the part of the remodel effect that does not run through quality (~$9.8k). The linear-quality caveat is a fair modelling worry, but it is secondary.
* Plumbing: `ames["Overall_Qual"].map(...)` worked because the index lines up with `remod_df`; `remod_df["Overall_Qual"].map(...)` is safer.

_What matched, what surprised you, and why (2-3 bullets):_
* My initial predictions were correct in the fact that remod's coefficient did shrink.
* My reasoning behind its value still holds I believe in relation to the signal it was trying to capture (implied quality).
* The shrink from adding quality was small: 11,387 to 9,791, only about **$1.6k** (14%). Most of the drop (34,257 to 11,387) came from the previous step, controlling for `Year_Built` and `Gr_Liv_Area`. Remodeled homes are only slightly higher quality on average (**6.6 vs 6.2**), so quality is a weak mediator in this data, not the whole story I predicted.

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Two correlated predictors in one model
*Role lean: DS · ML*

Garage capacity can be measured by number of cars or by square feet — two measures of nearly the same thing. Three models:

```text
corr(Garage_Cars, Garage_Area) = 0.89      VIF of each predictor in the two-variable model = 4.8

MODEL A: Sale_Price~Garage_Cars
                coef  std err     t   P>|t|
Intercept    60753.4   2842.6  21.4  <0.001
Garage_Cars  67966.2   1478.0  46.0  <0.001
   held-out RMSE (70/30 split) = 63,308

MODEL B: Sale_Price~Garage_Area
                coef  std err     t   P>|t|
Intercept    68470.4   2737.3  25.0  <0.001
Garage_Area    237.6      5.3  45.1  <0.001
   held-out RMSE (70/30 split) = 63,020

MODEL C: Sale_Price~Garage_Cars+Garage_Area
                coef  std err     t   P>|t|
Intercept    57526.7   2812.8  20.5  <0.001
Garage_Cars  39294.5   3185.1  12.3  <0.001
Garage_Area    114.0     11.3  10.1  <0.001
   held-out RMSE (70/30 split) = 61,932
```

(a) Both variables are strongly "significant" in Model C, yet the coefficients are about half of what they were alone. Say in plain words what the `Garage_Cars` coefficient of about 39,300 in Model C means versus the roughly 68,000 in Model A. (b) The standard error on `Garage_Area` more than doubled from Model B to Model C. Why? (c) Held-out RMSE barely moved across the three models. What does that tell you about when multicollinearity is and isn't a problem — for *prediction* versus *explaining which variable matters*? (d) A colleague says "VIF is only about 5, so there's no problem." What would you want to know before agreeing?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Why the coefficients wobble but the prediction doesn't
*Role lean: DS · ML · interview*

Continuing with Model C, here is how much its numbers jump around if you re-fit on bootstrap resamples of the data:

```text
1,000 bootstrap re-fits of MODEL C (Sale_Price ~ Garage_Cars + Garage_Area):
  std dev of the Garage_Cars coefficient across re-fits : 4,056  (full-data estimate 39,295)
  std dev of the Garage_Area coefficient across re-fits : 15.9  (full-data estimate 114.0)
  correlation between the two coefficient estimates     : -0.87
  predicted price for a 2-car, 500 sq ft garage: mean 193,153, std dev across re-fits 1,397
```

(a) The two coefficient estimates are strongly *negatively* correlated across re-fits. Explain intuitively why — what has to happen to the Area coefficient when the Cars coefficient bounces up? (b) Why is the *predicted price* so much more stable than either coefficient? (c) What does this say about whether you'd report "the effect of one more car stall" from this model to a stakeholder? What would you do instead (name two options)?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Predicting the sign of omitted-variable bias
*Role lean: Causal (supplementary) · DS · interview*

A short model leaves out `Year_Built`; a long model includes it. Everything below is from real fits:

```text
SHORT model: Sale_Price ~ Garage_Area
   Garage_Area coef = 237.6

LONG model:  Sale_Price ~ Garage_Area + Year_Built
   Garage_Area coef = 179.5     Year_Built coef = 861.2

Auxiliary:   Year_Built ~ Garage_Area
   Garage_Area coef = 0.0676   (each extra sq ft of garage goes with a house that is 0.0676 years newer, on average)
```

(a) Before looking at the numbers, which direction would you have predicted the bias to go (does the short-model coefficient on Garage_Area overstate or understate the long-model one), using only: "newer houses sell for more" and "newer houses have bigger garages"? (b) Verify the identity short = long + (Year_Built coef in long) × (Garage_Area coef in auxiliary) with the numbers above. (c) State the identity in words and use it to answer: if the omitted variable were *uncorrelated* with the one you kept, what would happen? If it were *negatively* correlated with the kept variable but positively related to price, which way would the bias go? (d) Why is this the formula behind most "controlling for X" conversations in causal inference?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — A control that hurts
*Role lean: Causal (supplementary) · DS · interview*

An agent wants to know what *adding a full bathroom* is worth. Here is the coefficient on `Full_Bath` as controls are added:

```text
Coefficient on Full_Bath (std err), by what else is in the model:
  Full_Bath only                         78,827  (2,238)
  + Year_Built                           52,530  (2,313)
  + Year_Built + Gr_Liv_Area             -8,856  (2,208)

Auxiliary:  Gr_Liv_Area ~ Full_Bath + Year_Built  ->  coefficient on Full_Bath = 606 sq ft per extra full bath
Full_Bath counts: 0 baths: 12, 1 baths: 1318, 2 baths: 1532, 3 baths: 64, 4 baths: 4
```

(a) Walk through the three coefficients: why does adding `Year_Built` shrink the number, and why does the last step flip the sign? (b) The auxiliary regression says each extra full bath goes with about 606 more square feet of living area. In causal-inference language, what role does `Gr_Liv_Area` play if the question is "what is a bathroom *addition* worth" — and what question does the third coefficient actually answer instead? (c) Which of the three numbers would you report for the agent's question, and which for a question like "among homes of identical size, does swapping a bedroom for a second bath raise price"? (d) Interview version: "You're told to include every available covariate to be safe. What could go wrong?" Give an example from this table.

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Reading an interaction
*Role lean: DS · Causal · interview*

A model that lets the price-per-square-foot differ between single-family homes and end-unit townhouses:

```text
Sale_Price ~ Gr_Liv_Area * C(Bldg_Type), only OneFam and TwnhsE homes
(reference level = OneFam)
                                       coef  std err     t   P>|t|
Intercept                            6094.2   3419.2   1.8   0.075
C(Bldg_Type)[T.TwnhsE]             -27118.1  17321.1  -1.6   0.118
Gr_Liv_Area                           117.7      2.1  55.3  <0.001
Gr_Liv_Area:C(Bldg_Type)[T.TwnhsE]     41.2     12.5   3.3   0.001

n = 2658   R-squared = 0.549   Adj. R-squared = 0.548   Residual std error = 54,801
Average Gr_Liv_Area: OneFam 1,518, TwnhsE 1,342
```

(a) What is the estimated price per extra sq ft for a OneFam home? For a TwnhsE home? (b) The `C(Bldg_Type)[T.TwnhsE]` coefficient is -27,118. Does that mean townhouses sell for $27k less? Where does that number apply, and is that a place you have data? (c) Compute the predicted TwnhsE-minus-OneFam price gap at 1,000 sq ft and at 2,500 sq ft. At roughly what size do the two fitted lines cross? (d) How would centering `Gr_Liv_Area` at its mean change the dummy coefficient's meaning, and what would stay the same? (e) The interaction has p = 0.001 but the TwnhsE main effect has p = 0.118. A teammate says "drop the non-significant main effect." Do you agree?

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_